# AI工学101 — 第27回

## 特徴量選択と次元削減：入力を「減らす」技術

第26回では、かなり実務寄りの前処理まで進みました。

```text
生データ
 ↓
欠損値補完
 ↓
数値の標準化
 ↓
カテゴリ変数のOne-Hot Encoding
 ↓
モデル
```

今回は、その次。

> **「そもそも、モデルに全部の特徴量を食わせる必要があるのか？」**

を考えます。

特徴量は増やせばいいわけではありません。

不要な特徴量や重複した情報が大量に入っていると、

* 過学習しやすくなる
* 計算量が増える
* モデルが解釈しにくくなる
* ノイズまで学習する
* データによっては性能が落ちる

ことがあります。

そこで今日は、

> **特徴量を選ぶ（Feature Selection）**

方法と、

> **特徴量そのものを別の低次元表現へ変換する（Dimensionality Reduction）**

方法を区別して学びます。

---

# 🎯 今日のゴール

今日できるようになること：

* 特徴量選択と次元削減の違いを説明できる
* `SelectKBest` を使える
* 特徴量とターゲットの統計的な関係を利用できる
* PCA（主成分分析）の基本的な考え方を理解する
* `PCA` をscikit-learnで実装できる
* Pipelineの中に特徴量選択・PCAを組み込める
* **「情報を減らすこと」も機械学習の設計になる**と理解する

---

# 📖 講義：約20〜25分

## 1. 特徴量が100個あったら全部使う？

例えば、

```text
特徴量1
特徴量2
特徴量3
...
特徴量100
```

があったとします。

でも実際には、

```text
特徴量3 → 重要

特徴量17 → 重要

特徴量42 → 重要

その他 → ほぼ無関係
```

かもしれません。

この場合、

```text
100個全部
```

ではなく、

```text
重要な10個
```

だけを使ったほうが良い可能性があります。

これが**特徴量選択（Feature Selection）**です。

---

# 🧠 2. 特徴量選択と次元削減は違う

ここは今日の重要ポイント。

## Feature Selection

**元の特徴量から選ぶ。**

例えば、

```text
age
income
height
weight
city
job
```

から、

```text
age
income
weight
```

だけを残す。

元の意味がそのまま残ります。

---

## Dimensionality Reduction

特徴量を**新しい軸に変換する**。

例えば、

```text
feature 1
feature 2
feature 3
feature 4
```

を、

```text
component 1
component 2
```

のように変換します。

つまり、

```text
特徴量を捨てる
```

のではなく、

> **情報をなるべく保ちながら、少ない次元に表現し直す**

という発想です。

---

# 🌳 3. 特徴量選択の方法

今日はまず、

**SelectKBest**

を使います。

名前の通り、

```text
K個の良い特徴量を選ぶ
```

方法です。

例えば、

```python
SelectKBest(k=5)
```

なら、

> 「10個ある特徴量から、スコアの高い5個を残して」

という意味です。

---

# 💻 実習1：Irisデータ

まずはおなじみのIris。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

形を確認。

```python
print(X.shape)
```

```text
(150, 4)
```

4特徴量です。

今回はこれを、

```text
4 → 2
```

に減らしてみます。

---

# 💻 実習2：SelectKBest

読み込み。

```python
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
```

作成。

```python
selector = SelectKBest(
    score_func=f_classif,
    k=2
)
```

変換。

```python
X_selected = selector.fit_transform(
    X,
    y
)
```

確認。

```python
print(
    X_selected.shape
)
```

```text
(150, 2)
```

4特徴量だったものが、

```text
2特徴量
```

になりました。

---

# 💻 実習3：どの特徴量が選ばれた？

```python
print(
    selector.get_support()
)
```

例えば、

```text
[False True True False]
```

のような結果になります。

これは、

```text
特徴量0 → 不採用

特徴量1 → 採用

特徴量2 → 採用

特徴量3 → 不採用
```

という意味です。

---

# 💻 実習4：特徴量名と対応させる

```python
for name, selected in zip(
    iris.feature_names,
    selector.get_support()
):
    print(
        name,
        selected
    )
```

これで、

> **どの元特徴量が選ばれたか**

を確認できます。

---

# 🧠 `f_classif` は何をしている？

今回使った、

```python
f_classif
```

は、特徴量とクラスとの関係を統計的に評価する方法の一つです。

ざっくり言えば、

> **この特徴量はクラスを区別するのに役立ちそうか？**

をスコア化しています。

---

# ⚠️ 重要：特徴量選択もtrainだけで行う

ここが第26回との接続。

本番では、

```text
全データ
 ↓
train / test
 ↓
trainで特徴量選択
 ↓
testには同じ選択を適用
```

とします。

全データを見て、

```python
selector.fit_transform(X, y)
```

してからtestを分けるのは避けます。

なぜなら、

**testの情報を使って特徴量を選んでしまう**

からです。

これもデータリークです。

---

# 💻 実習5：Pipelineに入れる

そこで、

```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
```

を使います。

```python
pipe = Pipeline([
    (
        "selector",
        SelectKBest(
            score_func=f_classif,
            k=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

学習。

```python
pipe.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = pipe.predict(
    X_test
)
```

評価。

```python
print(
    pipe.score(
        X_test,
        y_test
    )
)
```

これならCross Validationでも、

**各Foldのtrainデータだけから特徴量選択が行われます。**

---

# 💻 実習6：kを変える

```python
for k in [1, 2, 3, 4]:

    pipe = Pipeline([
        (
            "selector",
            SelectKBest(
                score_func=f_classif,
                k=k
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ])

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=5
    )

    print(
        "k =", k,
        "mean =", scores.mean()
    )
```

ここで、

```text
1特徴量
2特徴量
3特徴量
4特徴量
```

で性能を比較できます。

---

# 📖 4. もう一つの考え方：PCA

ここから次元削減。

**PCA（Principal Component Analysis / 主成分分析）**

です。

これは最初かなり不思議に見えるけど、直感を掴めば大丈夫。

---

# 🧠 PCAの直感

例えばデータが、

```text
      ●
    ●
  ●
●
```

のように斜め方向に並んでいるとします。

元の座標では、

```text
X1
X2
```

という2次元。

でも、データのばらつきはほとんど、

```text
斜め方向
```

にあります。

なら、

> **その斜め方向を新しい1本の軸として使えばよくない？**

という発想になります。

これがPCAの基本的な直感です。

---

# 🧠 5. PCAは「重要な方向」を探す

PCAは、

> **データの分散が大きい方向を新しい軸として見つける**

方法です。

その軸を、

```text
第1主成分
第2主成分
第3主成分
...
```

と呼びます。

そして、

```text
元の100次元
```

を

```text
主成分10個
```

などに変換できます。

---

# 💻 実習7：PCAを使う

```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
```

まず標準化。

```python
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X
)
```

PCA。

```python
pca = PCA(
    n_components=2
)
```

変換。

```python
X_pca = pca.fit_transform(
    X_scaled
)
```

形。

```python
print(
    X_pca.shape
)
```

```text
(150, 2)
```

4次元から2次元になりました。

---

# 💻 実習8：寄与率を見る

PCAで非常に重要なのが、

```python
pca.explained_variance_ratio_
```

です。

```python
print(
    pca.explained_variance_ratio_
)
```

例えば、

```text
[0.73, 0.23]
```

なら、

第1主成分が約73%、

第2主成分が約23%。

合計すると、

```text
約96%
```

です。

つまり、

> **4次元のデータを2次元にしたのに、元データのばらつきの約96%をこの2次元で表現できている**

という意味になります。

---

# 💻 実習9：累積寄与率

```python
print(
    pca.explained_variance_ratio_.cumsum()
)
```

例えば、

```text
[0.73, 0.96]
```

なら、

```text
1成分 → 73%

2成分 → 96%
```

です。

---

# 🧠 6. PCAで重要な注意

PCAは、

**「ターゲットを予測するのに最も役立つ方向」**

を探しているわけではありません。

基本的には、

> **入力データそのものの分散をよく説明する方向**

を探しています。

つまり、

```text
SelectKBest
```

と

```text
PCA
```

は目的が違います。

### SelectKBest

```text
ターゲットとの関係
↓
予測に役立ちそうな特徴量を選ぶ
```

### PCA

```text
入力データの構造
↓
情報を保ちやすい新しい軸を作る
```

ここはしっかり区別しましょう。

---

# 💻 実習10：PCAをPipelineに入れる

```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "pca",
        PCA(
            n_components=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

これで、

```text
元データ
 ↓
StandardScaler
 ↓
PCA
 ↓
2次元
 ↓
Logistic Regression
```

という流れになります。

---

# 💻 実習11：Cross Validation

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5
)

print(
    scores
)

print(
    scores.mean()
)
```

ここでも、

```text
Fold 1
 ↓
Scalerをtrainにfit
 ↓
PCAをtrainにfit
 ↓
Modelをtrainにfit
 ↓
validationで評価
```

となります。

つまりPCAの学習も、

**各Foldのtrainデータだけ**

で行われます。

---

# 🧠 7. SelectKBest vs PCA

今日の二つを並べるとかなり分かりやすい。

| 方法          | 何をする？     | 元の特徴量   |
| ----------- | --------- | ------- |
| SelectKBest | 良い特徴量を選ぶ  | 残る      |
| PCA         | 新しい軸に変換する | 直接は残らない |

例えば、

```text
age
income
height
weight
```

からSelectKBestなら、

```text
age
income
```

のように選べます。

PCAなら、

```text
PC1
PC2
```

という新しい特徴量になります。

---

# ✍️ 演習

## 問1

次の違いを説明してください。

```text
Feature Selection
Dimensionality Reduction
```

---

## 問2

`SelectKBest(k=2)` を使ってIrisの4特徴量から2特徴量を選んでください。

---

## 問3

`get_support()` を使って、どの特徴量が選ばれたか確認してください。

---

## 問4

PCAで、

```text
4次元 → 2次元
```

に変換してください。

---

## 問5

```python
explained_variance_ratio_
```

を表示し、

> 2主成分で元データの何%程度の分散を説明できたか

確認してください。

---

## 問6

次の2つをCross Validationで比較してください。

```text
Logistic Regression
```

と

```text
StandardScaler
↓
PCA(n_components=2)
↓
Logistic Regression
```

---

# 👾 ボス戦：特徴量削減パイプライン

今日の本丸。

Irisについて、次の3モデルを比較してください。

### A

```text
StandardScaler
↓
Logistic Regression
```

### B

```text
SelectKBest(k=2)
↓
Logistic Regression
```

### C

```text
StandardScaler
↓
PCA(n_components=2)
↓
Logistic Regression
```

それぞれについて、

```python
cross_val_score(
    ...,
    cv=5
)
```

を実行。

そして、

```text
平均Accuracy
標準偏差
```

を比較してください。

---

## ボス戦の問い

単純に、

> 「Accuracyが一番高いものが正解」

ではありません。

それぞれについて、

```text
・性能
・特徴量数
・解釈しやすさ
・情報の圧縮
```

を考えて、

**どんな状況ならSelectKBestが向いていて、どんな状況ならPCAが向いていそうか**

を説明してください。

---

# 🌱 今日のまとめ

今日の核心は、

> **特徴量は「増やす」だけでなく、「選ぶ」「変換する」という設計ができる。**

ということ。

これまで、

```text
特徴量エンジニアリング
        ↓
特徴量を作る
```

という方向を学びました。

今日は逆方向の、

```text
特徴量が多すぎる
        ↓
選ぶ
        ↓
減らす
        ↓
圧縮する
```

を学びました。

そして、

```text
SelectKBest
```

は、

> **元の特徴量から必要なものを選ぶ**

一方、

```text
PCA
```

は、

> **データを新しい低次元空間に写像する**

という違いがあります。

---

# 🧭 AI工学101・現在地

ここまで来ると、scikit-learn編の構造がかなり綺麗になっています。

```text
                 データ
                   ↓
              前処理
                   ↓
        ┌──────────┴──────────┐
        ↓                     ↓
   特徴量を作る           特徴量を選ぶ
        ↓                     ↓
 PolynomialFeatures      SelectKBest
                              ↓
                         PCA / 圧縮
                              ↓
                    ┌─────────┴─────────┐
                    ↓                   ↓
                  回帰                 分類
                                        ↓
                              ┌─────────┼─────────┐
                              ↓         ↓         ↓
                           Linear    Tree      Ensemble
                           Logistic  Random    Gradient
                                     Forest    Boosting
                                        ↓
                                    評価設計
                                        ↓
                                  CV / GridSearch
                                        ↓
                                  最終モデル
```

そして、この一連の工程を

```text
Pipeline
```

でつなげられる。

これがかなり重要な基礎体力になっています。

---

# 🔜 第28回

## モデルの中身を読む：係数・特徴量重要度・Permutation Importance

次回は、**「このAIは何を根拠に予測しているの？」**に踏み込みます。

扱うのは、

* Logistic Regressionの `coef_`
* Decision Tree / Random Forestの `feature_importances_`
* Permutation Importance
* 「重要度」と「因果関係」の違い
* モデル解釈の落とし穴
* 特徴量重要度をCross Validationとどう組み合わせるか

です。

ここまで来ると、単に「Accuracy 97%でした」で終わらず、

> **「このモデルは何を使って予測しているのか？」**

を調べられるようになります。これは後のAI研究・人間とAIの信頼性を考えるうえでも、かなり大事な土台になる。